# 23 — Advanced Internals: CPython Execution Model

Goal: understand what Python is doing under the hood (enough to debug weirdness and reason about performance).

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: CPython vs “Python”

Python is a language spec; CPython is the reference implementation.

Other implementations:
- PyPy (JIT)
- Jython (JVM)
- IronPython (.NET)
- MicroPython / CircuitPython (microcontrollers)

## 2.
L2: Bytecode and `dis`

Python source is compiled to bytecode, executed by the VM.
You can inspect bytecode with `dis`.

In [ ]:

import dis

def f(x):
    return x * 2 + 1

dis.dis(f)


## 3.
L3: Reference counting and garbage collection

CPython primarily uses reference counting, plus a cyclic GC for reference cycles.

In [ ]:

import sys, gc

a = []
print("refcount (approx):", sys.getrefcount(a))  # includes temp refs
a.append(a)  # create a reference cycle

print("gc enabled:", gc.isenabled())
unreachable = gc.collect()
print("gc collected:", unreachable)


## 4.
L4: The import system (very briefly)

Imports:
- locate module using `sys.meta_path` finders and `sys.path`
- create module object
- execute module code once
- cache in `sys.modules`

Most import bugs are project layout bugs.

## 5.
L5: Exception groups (Python 3.11+) (advanced)

Async and concurrent code can raise multiple exceptions together.
`ExceptionGroup` and `except*` handle them.

In [ ]:

def raise_group():
    raise ExceptionGroup("multiple", [ValueError("a"), TypeError("b")])

try:
    raise_group()
except* ValueError as eg:
    print("caught ValueError group:", eg)
except* TypeError as eg:
    print("caught TypeError group:", eg)


## 6.
L6: Exercises

1. Use `dis` on a list comprehension and compare to an explicit loop.
2. Create a cycle and observe `gc.collect()` behavior.
3. Explain why imports run code and why that matters for side effects.

## 7.
L7: Object sizes and `sys.getsizeof` (approximate)

`sys.getsizeof` returns the shallow size of an object (not including referenced objects).
Use it for rough intuition, not exact memory accounting.

In [ ]:

import sys
xs = list(range(1000))
print("list shallow size:", sys.getsizeof(xs))
print("int shallow size :", sys.getsizeof(0))


## 8.
L8: `inspect` (introspection for advanced debugging)

`inspect` can show source, signatures, stack frames, etc.

In [ ]:

import inspect

def demo(a, b=1): 
    return a + b

print(inspect.signature(demo))
print("source snippet:")
print("\n".join(inspect.getsource(demo).splitlines()))
